In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import yaml

epoch_better_path = "../../experiment_data/balance_metrics/balance_metrics_imagenet_1e-6.csv"

epoch_data = pd.read_csv(epoch_better_path)

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1
epoch_data.head()

,dataset,split,model,k,epoch,layer,thresh,thresh_mode,node_count,average_branching_factor,...,average_colless_index,total_colless,missing_colless,missing_colless_frac,sackin_index,average_sackin_index,leaf_count,total_volume,train_acc,val_acc
0,imagenet,trainUval,resnet50,20,0,apenultimate,0.000001,0.000001,16887,2.096598,...,61305.690135,7329,725,0.090017,1.861293e+09,210720.380052,8833,300000,0.023096,0.07076
1,imagenet,trainUval,resnet50,20,24,alayer3,0.000001,0.000001,24072,3.002120,...,34429.060161,5452,2566,0.320030,4.184149e+09,260629.696337,16054,300000,0.538175,0.55000
2,imagenet,trainUval,resnet50,20,40,apenultimate,0.000001,0.000001,6491,2.058357,...,46184.236559,2976,177,0.056137,7.550515e+08,226198.764230,3338,300000,0.691470,0.71488
3,imagenet,trainUval,resnet50,20,56,alayer2,0.000001,0.000001,19027,2.293394,...,25099.552203,6628,1668,0.201061,2.897132e+09,269977.838878,10731,300000,0.699842,0.70864
4,imagenet,trainUval,resnet50,20,8,aconv1,0.000001,0.000001,18079,2.242928,...,16556.139420,6857,1203,0.149256,2.709998e+09,270485.880727,10019,300000,0.465984,0.47740


In [8]:
epoch_trainUval = epoch_data[(epoch_data["split"] == "trainUval") & (epoch_data["layer"] == "apenultimate") & (epoch_data["model"] == "resnet50") & (epoch_data["dataset"] == "imagenet")]
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig1 = make_subplots(specs=[[{"secondary_y": True}]], )

train_acc = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", width=15, color_discrete_sequence=color_seq_train)
val_acc = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", width=15, color_discrete_sequence=color_seq_val)
sackin = px.scatter(epoch_trainUval, x='epoch', y='sackin_index', color="dataset", color_discrete_sequence=color_seq)

best_epoch = epoch_trainUval[epoch_trainUval["val_acc"] == epoch_trainUval["val_acc"].max()]["epoch"].values[0]

fig1.add_trace(sackin.data[0], secondary_y=False)
fig1.add_trace(train_acc.data[0], secondary_y=True)
fig1.add_trace(val_acc.data[0], secondary_y=True)

fig1.update_annotations(font_size=16)
fig1.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=400*2, height=180*2, font=dict(size=22), showlegend=False)
fig1.update_xaxes(title_text="Epoch", title_standoff=18, automargin=True, range=[0,84])
fig1.update_yaxes(title_text="Sackin Index", type="log", title_standoff=18, automargin=True, secondary_y=False)
fig1.update_yaxes(title_text="Accuracy", title_standoff=18, automargin=True, secondary_y=True, nticks=10, range=[0, 1.05])
fig1.write_image(f"resnet-imagenet-epoch.png", scale=8)
